# Retrieval-Augmented Generation (RAG) System for University Admissions

This project implements a RAG-based system to answer admissions-related questions using university data. The system combines document retrieval with large language models to generate accurate and grounded responses.

In [ ]:
import zipfile

with zipfile.ZipFile("data.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

print("Extracted!")

Extracted!


## 1. Data Collection

We collected admissions-related information from the University of New Haven website and saved the cleaned content as `.txt` files. These files were then loaded into Python for further processing.

In [ ]:
import os

documents = []
base_folder = "data/data"

for filename in os.listdir(base_folder):
    if filename.endswith(".txt"):
        file_path = os.path.join(base_folder, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
            documents.append({
                "source": filename,
                "text": text
            })

print("Number of documents loaded:", len(documents))
for doc in documents:
    print(doc["source"])

Number of documents loaded: 7
data:tuition_costs_grad.txt
data:undergrad_requirements.txt
grad_requirements.txt
data:international_requirements.txt
data:transfer_requirements.txt
deadlines.txt
tuition_costs_ug.txt


## 2. Data Preprocessing

To improve retrieval accuracy, the documents were split into smaller chunks. This allows the system to retrieve more precise and relevant pieces of information instead of entire documents.

In [ ]:
import re

def chunk_text(text, max_chunk_length=300):
    # Split into sections first
    sections = re.split(r'\n(?=[A-Z][A-Za-z’&/\-\s]+:)', text.strip())

    chunks = []

    for section in sections:
        section = section.strip()
        if not section:
            continue

        # Now split large sections into smaller pieces
        for i in range(0, len(section), max_chunk_length):
            chunk = section[i:i+max_chunk_length]
            chunks.append(chunk.strip())

    return chunks

In [ ]:
!pip install sentence-transformers

In [ ]:
chunked_documents = []

for doc in documents:
    chunks = chunk_text(doc["text"])
    for chunk in chunks:
        chunked_documents.append({
            "source": doc["source"],
            "text": chunk
        })

print("Total chunks:", len(chunked_documents))
print()
print(chunked_documents[0])

Total chunks: 71

{'source': 'data:tuition_costs_grad.txt', 'text': 'Graduate Tuition and Costs – University of New Haven (2026–2027)'}


## 3. Embedding Generation

We used the SentenceTransformer model (all-MiniLM-L6-v2) to convert text chunks into vector embeddings. These embeddings allow the system to perform semantic similarity search during retrieval.

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [doc["text"] for doc in chunked_documents]
embeddings = embed_model.encode(texts)

print("Embedding shape:", len(embeddings), len(embeddings[0]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: 71 384


## 4. Retrieval System

We implemented a cosine similarity-based retrieval system to find the most relevant text chunks for a given query. This allows the system to provide context-aware answers.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, top_k=3):
    query_embedding = embed_model.encode([query])
    similarities = cosine_similarity(query_embedding, embeddings)[0]

    query_lower = query.lower()
    results = []

    for idx, sim in enumerate(similarities):
        text = chunked_documents[idx]["text"]
        text_lower = text.lower()
        source = chunked_documents[idx]["source"].lower()

        if len(text.strip()) < 80:
            continue
        if "must meet additional requirements" in text_lower:
            continue

        bonus = 0

        if "deadline" in query_lower and "deadline" in text_lower:
            bonus += 0.25
        if "international" in query_lower and "international" in text_lower:
            bonus += 0.20

        # strong preference for the graduate deadlines file on deadline questions
        if "deadline" in query_lower and "deadlines" in source:
            bonus += 0.30

        # penalize misleading spring admission chunks from other files
        if "deadline" in query_lower and "spring admission" in text_lower and "deadlines" not in source:
            bonus -= 0.25

        final_score = sim + bonus

        results.append({
            "text": text,
            "score": final_score,
            "source": chunked_documents[idx]["source"]
        })

    results = sorted(results, key=lambda x: x["score"], reverse=True)
    return results[:top_k]

In [ ]:
results = retrieve("What is the deadline for international students?")
for r in results:
    print(r["text"])
    print("-"*40)

International Students:
- Fall Priority 1: May 1
- Spring Priority 1: October 15
----------------------------------------
Notes:
- Some programs have their own unique deadlines
- Applicants should check individual program pages for program-specific deadlines
----------------------------------------
Overview:
Graduate programs use rolling admissions, so students may still apply after the priority deadlines. Early application is recommended.
----------------------------------------


In [ ]:
questions = [
    "What is the priority deadline for international graduate students?",
    "What are the fall and spring application deadlines for graduate students?",
    "What documents are required for a graduate application?",
    "Do graduate applicants need GRE or GMAT scores?",
    "What English language test scores are required for international graduate students?",
    "What documents and test requirements are needed for international graduate applicants?",
    "What documents are required for transfer students applying to undergraduate programs?",
    "What is required for first-year undergraduate applicants?",
    "What is the total cost of attendance for undergraduate students living on campus?",
    "How much is graduate tuition per credit hour?"
]

In [ ]:
for q in questions:
    print("QUESTION:", q)

    results = retrieve(q, top_k=2)

    for r in results:
        print("Source:", r["source"])
        print("Score:", r["score"])
        print("Text:", r["text"][:200])  # preview
        print("---")

    print("=" * 60)

QUESTION: What is the priority deadline for international graduate students?
Source: deadlines.txt
Score: 1.163631
Text: International Students:
- Fall Priority 1: May 1
- Spring Priority 1: October 15
---
Source: deadlines.txt
Score: 1.1408947
Text: Overview:
Graduate programs use rolling admissions, so students may still apply after the priority deadlines. Early application is recommended.
---
QUESTION: What are the fall and spring application deadlines for graduate students?
Source: deadlines.txt
Score: 1.161519
Text: Overview:
Graduate programs use rolling admissions, so students may still apply after the priority deadlines. Early application is recommended.
---
Source: deadlines.txt
Score: 1.1332512
Text: Notes:
- Some programs have their own unique deadlines
- Applicants should check individual program pages for program-specific deadlines
---
QUESTION: What documents are required for a graduate application?
Source: grad_requirements.txt
Score: 0.63528806
Text: Overview:
Standard 

In [ ]:
for q in questions:
    print("QUESTION:", q)
    print("ANSWER:", rag_answer(q))
    print("=" * 60)

QUESTION: What is the priority deadline for international graduate students?


NameError: name 'rag_answer' is not defined

In [ ]:
!pip install transformers accelerate

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import whoami
print(whoami())

LocalTokenNotFoundError: Token is required to call the /whoami-v2 endpoint, but no token found. You must provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

In [ ]:
from transformers import AutoTokenizer
AutoTokenizer.from_pretrained("google/gemma-2b-it")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

GemmaTokenizer(name_or_path='google/gemma-2b-it', vocab_size=256000, model_max_length=1000000000000000019884624838656, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<bos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<eos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<bos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<mask>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<2mass>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	6: AddedToken("[@BOS@]", rstrip=False, lstrip=False, s

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b-it",
    device_map="auto"
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [ ]:
def rag_answer(query, top_k=3):

    # retrieve
    results = retrieve(query, top_k=top_k)
    context = "\n\n".join([r["text"] for r in results])

    # prompt
    prompt = f"""
You are a helpful university admissions assistant.

Answer in ONE complete sentence.
If there are multiple deadlines, include all of them explicitly with their labels (e.g., Fall, Spring).
Use ONLY exact information from the context.
Do not generalize or summarize beyond what is written.
Do not use bullet points or extra explanation.

Context:
{context}

Question:
{query}

Answer:
"""

    # tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.3,
        top_p=0.9
    )

    # Step 5 — decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # remove prompt from output
    answer_only = response[len(prompt):].strip()

    return answer_only

In [ ]:
print(rag_answer("What is the deadline for international students?"))


You are a helpful university admissions assistant.

Answer in ONE complete sentence.
If there are multiple deadlines, include all of them explicitly with their labels (e.g., Fall, Spring).
Use ONLY exact information from the context.
Do not generalize or summarize beyond what is written.
Do not use bullet points or extra explanation.

Context:
International Students:
- Fall Priority 1: May 1
- Spring Priority 1: October 15

Overview:
International applicants must meet standard graduate application requirements along with additional documentation and English language proficiency requirements.

Spring Admission:
- Rolling admissions
- Priority deadline: first week of December

Question:
What is the deadline for international students?

Answer:
The deadline for international students is May 1 for Fall Priority 1 and October 15 for Spring Priority 1.


In [ ]:
print(rag_answer("When should international students apply?"))


You are a helpful university admissions assistant.

Answer in ONE complete sentence.
If there are multiple deadlines, include all of them explicitly with their labels (e.g., Fall, Spring).
Use ONLY exact information from the context.
Do not generalize or summarize beyond what is written.
Do not use bullet points or extra explanation.

Context:
International Students:
- Fall Priority 1: May 1
- Spring Priority 1: October 15

Overview:
International applicants must meet standard graduate application requirements along with additional documentation and English language proficiency requirements.

Additional Notes:
- Student health insurance is mandatory for full-time international students
- Costs may vary based on course load and program
- Additional course or lab fees may apply
- All costs are subject to change

Question:
When should international students apply?

Answer:
International students must apply for Fall Priority 1 (May 1) or Spring Priority 1 (October 15).


In [ ]:
print(rag_answer("What is the spring deadline for international students?"))


You are a helpful university admissions assistant.

Answer in ONE complete sentence.
If there are multiple deadlines, include all of them explicitly with their labels (e.g., Fall, Spring).
Use ONLY exact information from the context.
Do not generalize or summarize beyond what is written.
Do not use bullet points or extra explanation.

Context:
International Students:
- Fall Priority 1: May 1
- Spring Priority 1: October 15

Spring Admission:
- Rolling admissions
- Priority deadline: first week of December

Fall Admission:
- Early Decision: December 1 (binding)
- Early Action I: December 15 (non-binding)
- Early Action II: February 15 (non-binding)
- Regular Decision: Rolling admissions (priority deadline March 1)

Question:
What is the spring deadline for international students?

Answer:
The spring deadline for international students is October 15.


In [ ]:
print(rag_answer("What is the tuition fee for international students?"))


You are a helpful university admissions assistant.

Answer in ONE complete sentence.
If there are multiple deadlines, include all of them explicitly with their labels (e.g., Fall, Spring).
Use ONLY exact information from the context.
Do not generalize or summarize beyond what is written.
Do not use bullet points or extra explanation.

Context:
Pre-Enrollment Fees:
- Application fee: $50
- Tuition deposit (domestic): $300
- Tuition deposit (international): $1,000

Tuition Structure:
- Graduate tuition: $1,185 per credit hour
- Full-time general fee: $280 per year
- Part-time general fee: $240 per year

Off-Campus Students:
- Tuition and fees: $51,270
- Total direct cost: $51,270

Indirect Costs (Estimated):

Question:
What is the tuition fee for international students?

Answer:
$1,000


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

small_model_name = "distilgpt2"

small_tokenizer = AutoTokenizer.from_pretrained(small_model_name)
small_tokenizer.pad_token = small_tokenizer.eos_token

small_model = AutoModelForCausalLM.from_pretrained(small_model_name).to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def rag_answer_small(query, top_k=3):

    # Step 1 — retrieve
    results = retrieve(query, top_k=top_k)
    context = "\n\n".join([r["text"] for r in results])

    # Step 2 — prompt
    prompt = f"""
You are a helpful university admissions assistant.

Answer in ONE complete sentence.
If there are multiple deadlines, include all of them explicitly with their labels (Fall, Spring).
Use ONLY the information provided.

Context:
{context}

Question:
{query}

Answer:
"""

    # Step 3 — tokenize
    inputs = small_tokenizer(prompt, return_tensors="pt").to(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    # Step 4 — generate
    outputs = small_model.generate(
        **inputs,
        max_new_tokens=100
    )

    # Step 5 — decode
    response = small_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Step 6 — remove prompt
    answer_only = response[len(prompt):].strip()

    return answer_only

In [ ]:
print(rag_answer(q))
print(rag_answer_small(q))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Fall Priority 1: May 1

Therefore, international students should apply by May 1 for the Fall Priority 1 deadline.
The priority deadline is the first week of December.
If you are a graduate student, you must apply for a full-time position.
If you are a graduate student, you must apply for a full-time position.
If you are a graduate student, you must apply for a full-time position.
If you are a graduate student, you must apply for a full-time position.
If you are a graduate student, you must apply for a full-time position.
